<a href="https://colab.research.google.com/github/Di-oss/com4/blob/%D0%A0%D0%B5%D0%B4%D0%B8%D0%BD%D0%B0/dev_Redina.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Модуль 4: Регистрация пользователей
# Интернет-магазин "Все и сразу"

from datetime import datetime
import uuid

class User:
    def __init__(self, login, email, password, name=""):
        self.id = str(uuid.uuid4())[:8]
        self.login = login
        self.email = email
        self.password = password
        self.name = name if name else login
        self.role = "Customer"
        self.reg_date = datetime.now()
        self.last_login = None
        self.status = "active"
        self.orders = []
        self.cart = None

    def authenticate(self, pwd):
        if self.password != pwd:
            return "wrong_password"
        if self.status != "active":
            return "inactive_user"
        self.last_login = datetime.now()
        return "success"

    def logout(self):
        print(f"До свидания, {self.name}!")

class Customer(User):
    def __init__(self, login, email, password, name=""):
        super().__init__(login, email, password, name)
        self.role = "Customer"
        self.loyalty_points = 0
        self.address = ""
        self.payment_methods = []

class Admin(User):
    def __init__(self, login, email, password, name=""):
        super().__init__(login, email, password, name)
        self.role = "Admin"
        self.permissions = ["all"]

# Хранилище
users = {}
users_by_login = {}
users_by_email = {}

def register():
    print("\n--- РЕГИСТРАЦИЯ НОВОГО ПОЛЬЗОВАТЕЛЯ ---")

    login = input("Логин: ")
    if login in users_by_login:
        print("Ошибка: этот логин уже занят")
        return None

    email = input("Email: ")
    if email in users_by_email:
        print("Ошибка: этот email уже зарегистрирован")
        return None

    password = input("Пароль: ")
    name = input("Имя (нажмите Enter, чтобы пропустить): ")

    user = Customer(login, email, password, name)
    users[user.id] = user
    users_by_login[login] = user
    users_by_email[email] = user

    print("\n--- РЕГИСТРАЦИЯ УСПЕШНА ---")
    print("ID:", user.id)
    print("Логин:", user.login)
    print("Имя:", user.name)
    print("Email:", user.email)
    print("Роль:", user.role)
    print("-----------------------------")

    return user

def login():
    print("\n--- ВХОД В СИСТЕМУ ---")

    login_or_email = input("Логин или Email: ")
    password = input("Пароль: ")

    user = users_by_login.get(login_or_email) or users_by_email.get(login_or_email)

    if user:
        auth_result = user.authenticate(password)
        if auth_result == "success":
            print("\n--- ВХОД ВЫПОЛНЕН ---")
            print("Добро пожаловать,", user.name)
            print("Ваша роль:", user.role)
            print("ID пользователя:", user.id)
            print("----------------------")
            return user
        elif auth_result == "wrong_password":
            print("Ошибка: неверный пароль. Попробуйте снова.")
            return None
        elif auth_result == "inactive_user":
            print("Ошибка: ваша учетная запись неактивна.")
            return None
    else:
        print("Ошибка: пользователь с таким логином или email не найден.")
        return None

def show_profile(user):
    print("\n--- ПРОФИЛЬ ПОЛЬЗОВАТЕЛЯ ---")
    print("ID:", user.id)
    print("Логин:", user.login)
    print("Email:", user.email)
    print("Имя:", user.name)
    print("Роль:", user.role)
    print("Статус:", "Активен" if user.status == "active" else "Заблокирован")
    print("Дата регистрации:", user.reg_date.strftime("%d.%m.%Y %H:%M"))

    if user.last_login:
        print("Последний вход:", user.last_login.strftime("%d.%m.%Y %H:%M"))

    if user.role == "Customer":
        print("Бонусные баллы:", user.loyalty_points)
        print("Адрес доставки:", user.address if user.address else "не указан")

    print("-----------------------------")

def list_all_users():
    print("\n--- СПИСОК ПОЛЬЗОВАТЕЛЕЙ ---")
    if not users:
        print("Нет зарегистрированных пользователей")
        return

    for i, user in enumerate(users.values(), 1):
        print(f"{i}. {user.login} | {user.name} | {user.email} | {user.role}")
    print("-------------------------------")

def add_delivery_address(user):
    address = input("Введите адрес доставки: ")
    user.address = address
    print("Адрес сохранен")

def add_loyalty_points(user):
    try:
        points = int(input("Введите количество баллов для начисления: "))
        user.loyalty_points += points
        print(f"Баллы начислены. Текущий баланс: {user.loyalty_points}")
    except ValueError:
        print("Ошибка: введите число")

# Тестовые данные
admin = Admin("admin", "admin@shop.ru", "admin123", "Администратор")
users[admin.id] = admin
users_by_login["admin"] = admin
users_by_email["admin@shop.ru"] = admin

test_user = Customer("ivan", "ivan@mail.ru", "12345", "Иван Петров")
users[test_user.id] = test_user
users_by_login["ivan"] = test_user
users_by_email["ivan@mail.ru"] = test_user

# Главный цикл
current_user = None

while True:
    print("\n" + "="*50)
    if current_user:
        print(f"ТЕКУЩИЙ ПОЛЬЗОВАТЕЛЬ: {current_user.name} [{current_user.role}]")
    else:
        print("ГЛАВНОЕ МЕНЮ")
    print("="*50)

    if not current_user:
        print("1 - Регистрация")
        print("2 - Вход")
        print("3 - Список пользователей")
        print("0 - Выход")
    else:
        print("1 - Просмотр профиля")
        print("2 - Выход из системы")
        if current_user.role == "Customer":
            print("3 - Добавить адрес доставки")
            print("4 - Начислить бонусные баллы")
        elif current_user.role == "Admin":
            print("3 - Список всех пользователей")
        print("0 - Выход из программы")

    choice = input("\nВыберите действие: ")

    if not current_user:
        if choice == "1":
            register()
        elif choice == "2":
            current_user = login()
        elif choice == "3":
            list_all_users()
        elif choice == "0":
            print("Программа завершена")
            break
        else:
            print("Неверный выбор. Попробуйте снова.")

    else:
        if choice == "1":
            show_profile(current_user)
        elif choice == "2":
            current_user.logout()
            current_user = None
        elif choice == "3" and current_user.role == "Customer":
            add_delivery_address(current_user)
        elif choice == "4" and current_user.role == "Customer":
            add_loyalty_points(current_user)
        elif choice == "3" and current_user.role == "Admin":
            list_all_users()
        elif choice == "0":
            print("Программа завершена")
            break
        else:
            print("Неверный выбор. Попробуйте снова.")


ГЛАВНОЕ МЕНЮ
1 - Регистрация
2 - Вход
3 - Список пользователей
0 - Выход

Выберите действие: 3

--- СПИСОК ПОЛЬЗОВАТЕЛЕЙ ---
1. admin | Администратор | admin@shop.ru | Admin
2. ivan | Иван Петров | ivan@mail.ru | Customer
-------------------------------

ГЛАВНОЕ МЕНЮ
1 - Регистрация
2 - Вход
3 - Список пользователей
0 - Выход

Выберите действие: 0
Программа завершена
